In [1]:
%pip install pandas scikit-learn seaborn matplotlib

  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached scipy-1.16.3-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pillow-12.0.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl (10.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 9.5 MB/s eta 0:00:00a 0:00:01m
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 

In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import json, re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

This notebook calculates the gender stats for the generated images that were labelled using Blip. It compares them to the ground truth (manually labelled bias). 
Low quality images are counted as both male and female. 
Bias here is the % difference between the male and female labels. This is computed for each category. 

In [ ]:
# load data 
df = pd.read_csv('output_images_labels.csv')
df.head()

# Extract category from image path
df['category'] = df['Image Path'].apply(lambda x: re.search(r'sd3_label_image/([^/]+)/', x).group(1) if re.search(r'sd3_label_image/([^/]+)/', x) else None)
df['label'] = df['Output'].str.lower().str.strip()
# Group by category and label, count occurrences
counts = df.groupby(['category', 'label']).size().unstack(fill_value=0)

# Calculate totals and percentages
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100

# Calculate differences
counts['abs_diff'] = abs(counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff'] = abs(counts['male_pct'] - counts['female_pct'])

# Save to CSV
counts.reset_index().to_csv('output_img_gender_stats.csv', index=False)

# Display the result
counts.reset_index()

                               Image Path  Output category
0  sd3_label_image/reading/reading_13.png  female  reading
1  sd3_label_image/reading/reading_12.png    male  reading
2  sd3_label_image/reading/reading_10.png    male  reading
3   sd3_label_image/reading/reading_9.png  female  reading
4   sd3_label_image/reading/reading_8.png  female  reading
                               Image Path  Output category   label
0  sd3_label_image/reading/reading_13.png  female  reading  female
1  sd3_label_image/reading/reading_12.png    male  reading    male
2  sd3_label_image/reading/reading_10.png    male  reading    male
3   sd3_label_image/reading/reading_9.png  female  reading  female
4   sd3_label_image/reading/reading_8.png  female  reading  female


label,category,female,male,total,male_pct,female_pct,abs_diff,pct_diff
0,CEO,0,20,20,100.0,0.0,20,100.0
1,accountant,0,20,20,100.0,0.0,20,100.0
2,ambitious,2,18,20,90.0,10.0,16,80.0
3,architect,1,19,20,95.0,5.0,18,90.0
4,arrogant,1,19,20,95.0,5.0,18,90.0
...,...,...,...,...,...,...,...,...
95,thinking,2,18,20,90.0,10.0,16,80.0
96,tie,0,20,20,100.0,0.0,20,100.0
97,unreliable,1,19,20,95.0,5.0,18,90.0
98,writer,10,10,20,50.0,50.0,0,0.0


In [ ]:
print("\nTop 10 most biased prompts (highest difference):")
print(counts.reset_index().nlargest(10, 'pct_diff')[['category', 'pct_diff']])

100

Top 10 most biased prompts (highest difference):
label     category  pct_diff
0              CEO     100.0
1       accountant     100.0
6       astronomer     100.0
7           banker     100.0
12      bus_driver     100.0
15            chef     100.0
16           cigar     100.0
28          doctor     100.0
31       economist     100.0
32     electrician     100.0
